# Conv1D Variational Autoencoder Latent Space Exploration for 1D Time Series

In this notebook we continue our exploration of Variational Autoencoders (VAEs), but shift the perspective from image data to time series signals.

Instead of working with 2D spatial structures such as images, we focus on 1D temporal data (Time Series Signals). For this purpose, we use 1D Convolutional Variational Autoencoders (Conv1D VAEs), which are specifically designed to process sequential data where one dimension represents time.


We will explore how **latent space size** and **KL weight (β)** affect a Conv1D Variational Autoencoder (VAE) trained on synthetic 1D time-series signals.

Steps:
1. Generate synthetic signals (1000 time bins, 5 Hz–12 kHz components, bursts, noise).
2. Train a Conv1D VAE for multiple latent dimensions.
3. Inspect losses and latent usage.
4. Visualise reconstructions and latent interpolations.
5. Compare original vs reconstructed spectra (FFT).

### Conv1D vs Conv2D

Convolutional neural networks differ depending on the structure of the input data.

#### Conv2D (images)
- Used for image data (height × width × channels)
- Filters move in two spatial directions (x and y)
- Detects spatial patterns such as edges, textures, shapes
- Example: object recognition in images

#### Conv1D (time series)
- Used for sequential data (time × channels)
- Filters move along a single axis (time)
- Detects temporal patterns such as oscillations, spikes, bursts, or trends
- Example: signal processing, audio, sensor data, physiological signals


In [ ]:
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, random_split
import matplotlib.pyplot as plt

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Using device:', device)

torch.manual_seed(42)
np.random.seed(42)

## Section 1: Synthetic dataset with labels

We generate 1D time series composed of random sinusoids plus optional bursts and noise.
We store a binary label `has_burst` indicating presence of a short high-frequency burst.

In [ ]:
# Synthetic data generator: mixture of sinusoids + optional bursts + noise
def generate_synthetic_signals(
    n_signals=5000,
    length=1000,
    fs=25_000.0,
    min_freq=5.0,
    max_freq=12_000.0,
    max_components=4,
    burst_prob=0.3,
    noise_std=0.1,
):
    t = np.arange(length) / fs
    all_signals = []
    has_burst = []
    rng = np.random.default_rng(42)
    for _ in range(n_signals):
        n_comp = rng.integers(1, max_components + 1)
        sig = np.zeros_like(t, dtype=np.float32)
        for _ in range(n_comp):
            f = rng.uniform(min_freq, max_freq)
            A = rng.uniform(0.5, 1.5)
            phi = rng.uniform(0, 2 * np.pi)
            sig += A * np.sin(2 * np.pi * f * t + phi)
        burst_flag = 0
        if rng.random() < burst_prob:
            burst_flag = 1
            burst_len = rng.integers(20, 80)
            start = rng.integers(0, length - burst_len)
            burst_t = t[:burst_len]
            fb = rng.uniform(1000, max_freq)
            Ab = rng.uniform(1.0, 2.0)
            sig[start:start+burst_len] += Ab * np.sin(2 * np.pi * fb * burst_t)
        sig += noise_std * rng.normal(size=length)
        all_signals.append(sig.astype(np.float32))
        has_burst.append(burst_flag)
    all_signals = np.stack(all_signals, axis=0)
    has_burst = np.array(has_burst, dtype=np.int64)
    return all_signals, has_burst

N = 6000
T = 1000
signals, labels_burst = generate_synthetic_signals(n_signals=N, length=T)
print('Raw signals shape:', signals.shape)
print('Raw labels shape:', labels_burst.shape, '| burst fraction:', labels_burst.mean())

mean = signals.mean()
std = signals.std() + 1e-8
signals = (signals - mean).astype(np.float32) / std
print('Normalized signals shape:', signals.shape, 'mean≈', signals.mean(), 'std≈', signals.std())


In [ ]:
import numpy as np
import matplotlib.pyplot as plt

def plot_examples(signals, labels, fs=25_000, n=5):
    t = np.arange(signals.shape[1]) / fs

    idx_normal = np.where(labels == 0)[0][:n]
    idx_burst = np.where(labels == 1)[0][:n]

    fig, axes = plt.subplots(2, n, figsize=(15, 5), sharex=True)

    for i, idx in enumerate(idx_normal):
        axes[0, i].plot(t, signals[idx])
        axes[0, i].set_title("Normal")
        axes[0, i].set_yticks([])

    for i, idx in enumerate(idx_burst):
        axes[1, i].plot(t, signals[idx])
        axes[1, i].set_title("Burst")
        axes[1, i].set_yticks([])

    axes[0, 0].set_ylabel("Normal")
    axes[1, 0].set_ylabel("Burst")

    plt.tight_layout()
    plt.show()

plot_examples(signals, labels_burst)

## Section 2. Dataset and DataLoaders

We wrap signals in a simple `Dataset`, split into train/validation, and keep track of indices
so that labels can be aligned with latent codes later.

In [ ]:
class TimeSeriesDataset(Dataset):
    def __init__(self, data: np.ndarray):
        self.data = torch.from_numpy(data)
    def __len__(self):
        return self.data.shape[0]
    def __getitem__(self, idx):
        return self.data[idx]

dataset = TimeSeriesDataset(signals)
val_frac = 0.2
val_size = int(len(dataset) * val_frac)
train_size = len(dataset) - val_size
train_ds, val_ds = random_split(dataset, [train_size, val_size])
batch_size = 128
train_loader = DataLoader(train_ds, batch_size=batch_size, shuffle=True)
val_loader   = DataLoader(val_ds, batch_size=batch_size, shuffle=False)
print('Train size:', len(train_ds), '| Val size:', len(val_ds))

train_indices = train_ds.indices if hasattr(train_ds, 'indices') else np.arange(len(train_ds))
val_indices   = val_ds.indices if hasattr(val_ds, 'indices') else np.arange(len(val_ds))
labels_burst_train = labels_burst[train_indices]
labels_burst_val   = labels_burst[val_indices]

## Section 3: Conv1D VAE architecture

Encoder: Conv1D stack with stride 2, followed by a dense bottleneck to μ and logσ².
Decoder: dense layer back to feature map, then ConvTranspose1D stack to reconstruct the signal.

In [ ]:
class ConvVAE(nn.Module):
    def __init__(self, input_length=1000, latent_dim=32):
        super().__init__()
        self.input_length = input_length
        self.latent_dim = latent_dim
        self.enc_conv = nn.Sequential(
            nn.Conv1d(1, 32, kernel_size=9, stride=2, padding=4),
            nn.ReLU(),
            nn.Conv1d(32, 64, kernel_size=9, stride=2, padding=4),
            nn.ReLU(),
            nn.Conv1d(64, 128, kernel_size=9, stride=2, padding=4),
            nn.ReLU(),
        )
        conv_out_channels = 128
        conv_out_length   = 125
        enc_flat_dim      = conv_out_channels * conv_out_length
        self.enc_fc = nn.Sequential(
            nn.Linear(enc_flat_dim, 512),
            nn.ReLU(),
        )
        self.fc_mu     = nn.Linear(512, latent_dim)
        self.fc_logvar = nn.Linear(512, latent_dim)
        self.dec_fc = nn.Sequential(
            nn.Linear(latent_dim, 512),
            nn.ReLU(),
            nn.Linear(512, enc_flat_dim),
            nn.ReLU(),
        )
        self.dec_deconv = nn.Sequential(
            nn.ConvTranspose1d(128, 64, kernel_size=9, stride=2, padding=4, output_padding=1),
            nn.ReLU(),
            nn.ConvTranspose1d(64, 32, kernel_size=9, stride=2, padding=4, output_padding=1),
            nn.ReLU(),
            nn.ConvTranspose1d(32, 1, kernel_size=9, stride=2, padding=4, output_padding=1),
        )
    def encode(self, x):
        h = self.enc_conv(x)
        h = h.view(h.size(0), -1)
        h = self.enc_fc(h)
        mu = self.fc_mu(h)
        logvar = self.fc_logvar(h)
        return mu, logvar
    def reparameterize(self, mu, logvar):
        std = torch.exp(0.5 * logvar)
        eps = torch.randn_like(std)
        return mu + eps * std
    def decode(self, z):
        h = self.dec_fc(z)
        h = h.view(h.size(0), 128, 125)
        return self.dec_deconv(h)
    def forward(self, x):
        mu, logvar = self.encode(x)
        z = self.reparameterize(mu, logvar)
        x_rec = self.decode(z)
        return x_rec, mu, logvar

## Section 4. VAE loss and training loop

Loss = reconstruction MSE + β · KL divergence between q(z|x) and N(0, I).

In [ ]:
def vae_loss_function(recon_x, x, mu, logvar, beta=1e-3):
    recon_loss = F.mse_loss(recon_x, x, reduction='mean')
    kl_per_sample = -0.5 * torch.sum(1 + logvar - mu.pow(2) - logvar.exp(), dim=1)
    kl_loss = kl_per_sample.mean()
    total_loss = recon_loss + beta * kl_loss
    return total_loss, recon_loss.detach(), kl_loss.detach()

def train_one_vae(latent_dim,
                  train_loader,
                  val_loader,
                  input_length=1000,
                  epochs=20,
                  lr=1e-3,
                  beta=1e-3):
    model = ConvVAE(input_length=input_length, latent_dim=latent_dim).to(device)
    opt = torch.optim.Adam(model.parameters(), lr=lr)
    history = {
        'train_total': [], 'train_recon': [], 'train_kl': [],
        'val_total': [], 'val_recon': [], 'val_kl': [],
    }
    for ep in range(1, epochs + 1):
        model.train()
        train_total = train_recon = train_kl = 0.0
        for batch in train_loader:
            batch = batch.to(device).unsqueeze(1)
            opt.zero_grad()
            recon, mu, logvar = model(batch)
            loss, recon_l, kl_l = vae_loss_function(recon, batch, mu, logvar, beta=beta)
            loss.backward()
            opt.step()
            bs = batch.size(0)
            train_total += loss.item() * bs
            train_recon += recon_l.item() * bs
            train_kl    += kl_l.item() * bs
        n_train = len(train_loader.dataset)
        train_total /= n_train
        train_recon /= n_train
        train_kl    /= n_train
        model.eval()
        val_total = val_recon = val_kl = 0.0
        with torch.no_grad():
            for batch in val_loader:
                batch = batch.to(device).unsqueeze(1)
                recon, mu, logvar = model(batch)
                loss, recon_l, kl_l = vae_loss_function(recon, batch, mu, logvar, beta=beta)
                bs = batch.size(0)
                val_total += loss.item() * bs
                val_recon += recon_l.item() * bs
                val_kl    += kl_l.item() * bs
        n_val = len(val_loader.dataset)
        val_total /= n_val
        val_recon /= n_val
        val_kl    /= n_val
        history['train_total'].append(train_total)
        history['train_recon'].append(train_recon)
        history['train_kl'].append(train_kl)
        history['val_total'].append(val_total)
        history['val_recon'].append(val_recon)
        history['val_kl'].append(val_kl)
        print(
            f"[VAE latent={latent_dim:3d}, beta={beta:.0e}] Ep {ep:3d} | "
            f"train total {train_total:.5f} (recon {train_recon:.5f}, KL {train_kl:.5f}) | "
            f"val total {val_total:.5f} (recon {val_recon:.5f}, KL {val_kl:.5f})"
        )
    return model, history

## Section 5: Latent dimension sweep

Train ConvVAE for several latent dimensions and compute latent usage (Var[μ]).

### Latent usage (variance of μ)

Latent usage is a way to measure how much each dimension of the latent space in a Variational Autoencoder (VAE) is actually being used to encode information from the data.

In a VAE, each input is encoded not as a single point, but as a distribution parameterized by a mean vector (μ) and variance (logvar). The mean μ represents the central tendency of the encoded representation for each sample.

To estimate latent usage, we compute the variance of μ across the dataset for each latent dimension:

- High variance of μ in a given dimension means that this dimension changes significantly across different inputs → it carries meaningful information.
- Low variance of μ means that the dimension is nearly constant across samples → it is not being used effectively by the model.

Therefore, latent usage tells us which parts of the latent space are active and informative, and which are redundant or unused.

This is especially important in VAEs because the KL divergence term encourages the model to use as few latent dimensions as necessary, potentially "turning off" unused ones.

In [ ]:
latent_dims_vae = [2, 4, 8, 16, 32, 64, 128]
beta_default = 1e-3
results_vae = {}
for ld in latent_dims_vae:
    print('=' * 80)
    print(f'Training ConvVAE with latent_dim = {ld}, beta = {beta_default}')
    model, history = train_one_vae(
        latent_dim=ld,
        train_loader=train_loader,
        val_loader=val_loader,
        input_length=signals.shape[1],
        epochs=20,
        lr=1e-3,
        beta=beta_default,
    )
    model.eval()
    mus = []
    with torch.no_grad():
        for batch in val_loader:
            batch = batch.to(device).unsqueeze(1)
            mu, logvar = model.encode(batch)
            mus.append(mu.cpu().numpy())
    mus = np.concatenate(mus, axis=0)
    latent_var_mu = mus.var(axis=0)
    results_vae[ld] = {
        'model': model,
        'history': history,
        'val_latent_mu_var': latent_var_mu,
    }




## Section 6: MSE vs Latent dimension and latent dimension usage

Plot validation reconstruction MSE vs latent dimension and Var[μ] per coordinate.

### Elbow plot (reconstruction error vs latent dimension)

An "elbow" plot is a diagnostic tool used to analyze how the performance of a model changes as we increase its capacity - in this case, the latent dimensionality of a Variational Autoencoder (VAE).

We plot the final validation reconstruction error (e.g., MSE) as a function of the latent dimension.

A typical "Elbow" plot:

- At small latent dimensions, increasing the size of the latent space significantly improves reconstruction quality because the model gains more capacity to represent the data.
- After a certain point, increasing latent dimensionality leads to diminishing returns, and the reconstruction error stops improving significantly.

This point of transition is called the "elbow". It indicates the optimal trade-off between model complexity and reconstruction performance.

In the context of VAEs, the elbow plot helps estimate the intrinsic dimensionality of the data, i.e., how many latent variables are actually needed to represent the underlying structure of the signals.

In [ ]:
val_recon_final_vae = [results_vae[ld]['history']['val_recon'][-1] for ld in latent_dims_vae]
plt.figure()
plt.plot(latent_dims_vae, val_recon_final_vae, marker='o')
plt.xscale('log', base=2)
plt.xlabel('Latent dimension')
plt.ylabel('Final validation reconstruction MSE')
plt.title('ConvVAE: validation reconstruction error vs latent dim')
plt.grid(True)
plt.show()

In [ ]:
plt.figure(figsize=(10, 6))
for ld in latent_dims_vae:
    var_mu = results_vae[ld]['val_latent_mu_var']
    plt.plot(range(ld), var_mu, marker='o', label=f'latent={ld}')
plt.xlabel('Latent coordinate index')
plt.ylabel('Var(mu) over validation set')
plt.yscale('log')
plt.title('ConvVAE: latent usage (variance of μ)')
plt.legend()
plt.grid(True)
plt.show()

## Section 7: Time-domain reconstructions

Choose a latent dimension (here 16), visualise original vs reconstructed signals and latent interpolations.

---
In this step, we evaluate the trained ConvVAE in two ways: reconstruction quality and latent space structure.


### 1. Reconstructions

We compare original time series signals with their reconstructions from the model.

Pipeline:
- encode signal → latent distribution (μ, logvar)
- sample latent vector z
- decode z → reconstructed signal

#### Goal:
Check how well the model preserves the original signal structure.

- Good reconstruction → model captures important temporal patterns
- Poor reconstruction → information loss or underpowered latent space



In [ ]:
def plot_vae_reconstructions(model, dataset, n_examples=5, title_prefix=''):
    model.eval()
    idxs = np.random.choice(len(dataset), size=n_examples, replace=False)
    x_batch = torch.stack([dataset[i] for i in idxs]).to(device)
    with torch.no_grad():
        x_in = x_batch.unsqueeze(1)
        x_rec, mu, logvar = model(x_in)
        x_rec = x_rec.squeeze(1).cpu().numpy()
    x_batch = x_batch.cpu().numpy()
    plt.figure(figsize=(10, 2 * n_examples))
    for i in range(n_examples):
        plt.subplot(n_examples, 1, i + 1)
        plt.plot(x_batch[i], label='original', alpha=0.7)
        plt.plot(x_rec[i], label='reconstruction', alpha=0.7)
        plt.legend(loc='upper right')
        plt.xlabel('Time bin')
        plt.ylabel('Amplitude (norm.)')
    plt.suptitle(f"{title_prefix} ConvVAE: original vs reconstruction")
    plt.tight_layout()
    plt.show()

chosen_ld = 128
model_chosen = results_vae[chosen_ld]['model']
plot_vae_reconstructions(model_chosen, val_ds,
                         n_examples=5,
                         title_prefix=f'latent_dim={chosen_ld}')


## Section 8. Frequency-domain diagnostics (FFT)

Compare FFT magnitude of original vs reconstructed signals to see which frequencies are preserved.

---

In this step, we analyze how well the ConvVAE preserves frequency information in the reconstructed signals.

Instead of looking at signals in the time domain, we transform them into the frequency domain using the Fast Fourier Transform (FFT).

---

### Procedure

For each signal:
- Compute FFT of the original signal
- Compute FFT of the reconstructed signal
- Compare their magnitude spectra

$$
X(f) = \mathcal{F}\{x(t)\}
$$

We then compare:
- $|X_{\text{original}}(f)|$
- $|X_{\text{reconstructed}}(f)|$

---

### Goal

Evaluate which frequency components are preserved or lost during reconstruction.

- Matching spectra → model preserves signal structure
- Missing or damped frequencies → information loss in latent representation

---

### Why this matters

Time-domain reconstructions can look similar even if frequency content is distorted.

FFT reveals:
- whether the model captures high-frequency bursts
- whether it smooths out fine-grained structure
- how much spectral information is retained in latent space

---

### Interpretation

- Low frequencies → usually well preserved (global structure)
- High frequencies → often reduced or smoothed by the model
- Differences in spectra → indicate what information the latent space cannot fully encode


In [ ]:
def plot_fft_comparison(model, dataset, fs=25_000.0, n_examples=3, title_prefix=''):
    model.eval()
    idxs = np.random.choice(len(dataset), size=n_examples, replace=False)
    x_batch = torch.stack([dataset[i] for i in idxs]).to(device)
    with torch.no_grad():
        x_in = x_batch.unsqueeze(1)
        x_rec, mu, logvar = model(x_in)
        x_rec = x_rec.squeeze(1).cpu().numpy()
    x_batch = x_batch.cpu().numpy()
    T = x_batch.shape[1]
    freqs = np.fft.rfftfreq(T, d=1.0/fs)
    plt.figure(figsize=(10, 3 * n_examples))
    for i in range(n_examples):
        X_orig = np.fft.rfft(x_batch[i])
        X_rec  = np.fft.rfft(x_rec[i])
        plt.subplot(n_examples, 1, i + 1)
        plt.semilogy(freqs, np.abs(X_orig) + 1e-12, label='original')
        plt.semilogy(freqs, np.abs(X_rec) + 1e-12, label='reconstruction')
        plt.xlim(0, 12000)
        plt.xlabel('Frequency [Hz]')
        plt.ylabel('|X(f)|')
        plt.legend(loc='upper right')
    plt.suptitle(f"{title_prefix} FFT magnitude: original vs reconstruction")
    plt.tight_layout()
    plt.show()

plot_fft_comparison(model_chosen, val_ds,
                    fs=25_000.0,
                    n_examples=3,
                    title_prefix=f'latent_dim={chosen_ld}')

## Exploration Tasks (Basic)


### 0. Try to improve the results 🙃

### 1. Understand the elbow plot
- Which latent dimension gives the best result?
- Does increasing latent size always help?
- What does this say about signal complexity?

---

### 2. Explore latent usage
- Are all latent dimensions used?
- Which dimensions have very low `Var(μ)`?
- Do some latent variables become "dead"?

---

### 3. Simplify the dataset
Try:
- one sinusoid only
- no bursts
- less noise
- smaller frequency range

Observe how this changes:
- reconstruction error
- latent usage
- elbow plot shape

---

### 4. Make the dataset harder
Try:
- more sinusoids
- more bursts
- higher noise
- frequency drift

Questions:
- Does the optimal latent dimension increase?
- Does reconstruction become harder?

---
